# InsuML benchmark — scikit-learn baseline

Time-boxed model search over a small set of standard scikit-learn
estimators (logistic/ridge regression, random forest, histogram-based
gradient boosting, SVM), following the same general approach as
AdaptoML-UX (Gomaa et al., 2024).

Requires: `requirements.txt` (scikit-learn 1.9.1).
Shares its experimental protocol (task list, folds, seeds) with
`ludwig_experiment.ipynb` via `common_utils.py`, so results from both
notebooks are directly comparable.

In [ ]:
import os
import sys

# common_utils.py lives in the parent folder (shared by both notebooks)
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
import time
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import roc_auc_score, accuracy_score, mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import (
    RandomForestClassifier, RandomForestRegressor,
    HistGradientBoostingClassifier, HistGradientBoostingRegressor,
)
from sklearn.svm import SVC, SVR

from common_utils import (
    TASK_IDS, TEST_MODE, RUN_MODE_NAME, TIME_LIMIT_S,
    get_fold_repeat_seed_plan,
    save_environment_info,
    load_completed_runs,
    RESUME,
    results_dir,
    start_fresh_if_requested,
    DATASET_RETRIES,
    archive_task_results,
    extra_classification_metrics,
    download_openml_task_split,
    safe_metric_value,
    is_binary_classification,
    save_incremental_results,
    save_summary,
)

import openml

In [ ]:
# Time budget per scikit-learn search (seconds)
SKLEARN_TIME_BUDGET = TIME_LIMIT_S  # set in common_utils.py

SKLEARN_N_ITER_PER_MODEL = 15  # candidate configs tried per model family
SKLEARN_INNER_CV = 3           # inner CV folds used by RandomizedSearchCV

## Model search space

In [ ]:
CLASSIFIER_SEARCH_SPACE = {
    "logreg": (
        LogisticRegression,
        {"C": [0.01, 0.1, 1, 10, 100], "max_iter": [2000]},
    ),
    "random_forest": (
        RandomForestClassifier,
        {"n_estimators": [100, 200, 400], "max_depth": [None, 5, 10, 20]},
    ),
    "hist_gb": (
        HistGradientBoostingClassifier,
        {"max_iter": [100, 200, 300], "max_depth": [None, 5, 10]},
    ),
    "svc": (
        SVC,
        {"C": [0.1, 1, 10], "kernel": ["rbf", "linear"], "probability": [True]},
    ),
}

REGRESSOR_SEARCH_SPACE = {
    "ridge": (Ridge, {"alpha": [0.01, 0.1, 1.0, 10.0, 100.0]}),
    "random_forest": (
        RandomForestRegressor,
        {"n_estimators": [100, 200, 400], "max_depth": [None, 5, 10, 20]},
    ),
    "hist_gb": (
        HistGradientBoostingRegressor,
        {"max_iter": [100, 200, 300], "max_depth": [None, 5, 10]},
    ),
    "svr": (SVR, {"C": [0.1, 1, 10], "kernel": ["rbf", "linear"]}),
}

## Preprocessing and experiment functions

In [ ]:
def build_preprocessor(attribute_names, categorical_indicator):
    """
    Routes numeric and categorical columns to separate preprocessing
    pipelines, using OpenML's `categorical_indicator` rather than
    inferring column type from pandas dtypes.
    """
    numeric_cols = [
        name for name, is_cat in zip(attribute_names, categorical_indicator) if not is_cat
    ]
    categorical_cols = [
        name for name, is_cat in zip(attribute_names, categorical_indicator) if is_cat
    ]

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", StandardScaler()),
    ])
    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    return ColumnTransformer(transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ])


def run_sklearn_experiment(train_df, test_df, target_column, seed, task_type,
                            attribute_names, categorical_indicator, time_budget):
    """
    Time-boxed model selection over a small set of standard scikit-learn
    estimators. Iterates through model families in order, stopping once
    the time budget is exhausted; the best-scoring configuration across
    all families tried is refit and evaluated on the held-out test set.
    """
    is_classification = "classification" in task_type.lower()
    space = CLASSIFIER_SEARCH_SPACE if is_classification else REGRESSOR_SEARCH_SPACE

    X_train = train_df.drop(columns=[target_column])
    y_train = train_df[target_column]
    X_test = test_df.drop(columns=[target_column])
    y_test = test_df[target_column]

    preprocessor = build_preprocessor(
        [c for c in attribute_names if c != target_column]
        if target_column in attribute_names else attribute_names,
        categorical_indicator,
    )

    best_score = -np.inf
    best_pipeline = None
    best_model_name = None

    start_time = time.time()
    for model_name, (model_cls, param_grid) in space.items():
        elapsed = time.time() - start_time
        if elapsed >= time_budget:
            print(f"[INFO] Time budget exhausted, skipping \'{model_name}\' and the rest.")
            break

        try:
            pipeline = Pipeline(steps=[
                ("preprocessor", preprocessor),
                ("model", model_cls(random_state=seed) if "random_state" in model_cls().get_params() else model_cls()),
            ])
            prefixed_grid = {f"model__{k}": v for k, v in param_grid.items()}

            search = RandomizedSearchCV(
                pipeline,
                param_distributions=prefixed_grid,
                n_iter=SKLEARN_N_ITER_PER_MODEL,
                cv=SKLEARN_INNER_CV,
                random_state=seed,
                n_jobs=-1,
                scoring="roc_auc" if (is_classification and is_binary_classification(task_type, y_train))
                else ("accuracy" if is_classification else "neg_root_mean_squared_error"),
                error_score=np.nan,
            )
            search.fit(X_train, y_train)

            if search.best_score_ is not None and search.best_score_ > best_score:
                best_score = search.best_score_
                best_pipeline = search.best_estimator_
                best_model_name = model_name

        except Exception as e:
            print(f"[WARN] Training failed for \'{model_name}\': {e}")
            continue

    if best_pipeline is None:
        print("[ERROR] No scikit-learn model trained successfully for this task.")
        return None

    metrics = {}
    try:
        if is_classification:
            if is_binary_classification(task_type, y_train):
                y_proba = best_pipeline.predict_proba(X_test)[:, 1]
                metrics["roc_auc"] = safe_metric_value(roc_auc_score(y_test, y_proba))
            y_pred = best_pipeline.predict(X_test)
            metrics["accuracy"] = safe_metric_value(accuracy_score(y_test, y_pred))
            metrics.update(extra_classification_metrics(y_test, y_pred))
        else:
            y_pred = best_pipeline.predict(X_test)
            mse = mean_squared_error(y_test, y_pred)
            metrics["mean_absolute_error"] = safe_metric_value(mean_absolute_error(y_test, y_pred))
            metrics["mean_squared_error"] = safe_metric_value(mse)
            metrics["r2"] = safe_metric_value(r2_score(y_test, y_pred))
            metrics["root_mean_squared_error"] = safe_metric_value(
                np.sqrt(mse)
            )
        metrics["best_model"] = best_model_name
    except Exception as e:
        print(f"[ERROR] Evaluating the best scikit-learn model: {e}")
        return None

    return metrics

## Main loop

In [ ]:
# results/<run mode>/sklearn/ (next to analysis/ and logs/); see common_utils.py
save_dir = results_dir("sklearn")
start_fresh_if_requested(save_dir)  # only acts with BENCH_RESUME=0
os.makedirs(save_dir, exist_ok=True)
save_environment_info(save_dir, "sklearn")

all_task_ids = TASK_IDS[:1] if TEST_MODE else TASK_IDS

for task_id in all_task_ids:
    print(f"\n=== Processing task {task_id} ===")
    try:
        task = openml.tasks.get_task(task_id)
    except Exception as e:
        print(f"[ERROR] Could not fetch task {task_id}: {e}")
        continue

    plan = get_fold_repeat_seed_plan(task)
    # A finished dataset is skipped; a half-done one is discarded and restarted from fold 0
    runs, (dataset_name, task_type, evaluation_measure) = load_completed_runs(
        save_dir, task_id, "sklearn", expected_runs=len(plan))

    for attempt in range(1 + DATASET_RETRIES):
        if len(runs) == len(plan):
            break
        if attempt > 0:
            print(f"[RETRY] task {task_id}: a run failed; discarding this dataset's runs "
                  f"and starting it over (attempt {attempt + 1} of {1 + DATASET_RETRIES})")
            archive_task_results(save_dir, task_id, "sklearn")
            runs = []
        for (repeat, fold, seed) in plan:
            split_data = download_openml_task_split(task, repeat, fold)
            if split_data is None:
                break

            dataset_name = split_data["dataset_name"]
            task_type = split_data["task_type"]
            evaluation_measure = split_data["evaluation_measure"]

            print(f"[RUN] task={task_id} repeat={repeat} fold={fold} seed={seed}")

            run_start = time.perf_counter()
            metrics = run_sklearn_experiment(
                split_data["train_df"], split_data["test_df"],
                split_data["target_column"], seed, task_type,
                split_data["attribute_names"], split_data["categorical_indicator"],
                SKLEARN_TIME_BUDGET,
            )
            run_time = time.perf_counter() - run_start
            if not metrics:
                break  # a failed run: the whole dataset is retried
            metrics["time_taken"] = round(run_time, 2)  # seconds, train + eval
            runs.append({"repeat": repeat, "fold": fold, "seed": seed, "metrics": metrics})
            save_incremental_results(
                task_id, "sklearn", runs, dataset_name, task_type, evaluation_measure, save_dir,
            )
    if len(runs) < len(plan):
        print(f"[WARN] task {task_id}: only {len(runs)} of {len(plan)} runs completed after "
              f"{1 + DATASET_RETRIES} attempt(s). Launch the benchmark again to restart it.")

    save_summary(save_dir, "sklearn")

print("\nAll tasks processed.")
print(f"Results in: {save_dir}")

## Analysis

Summary table and statistics from the results above (the same as `python analyze_results.py <mode>`). Edit `ANALYSIS_OUTPUTS` to choose which files are written.

In [ ]:
# Which analysis files to write; the summary table is always shown below.
# Available: summary_table, baseline, time_and_failures, ludwig_diagnostics,
# per_task_stats, paired_tests (the last needs both tools' results).
ANALYSIS_OUTPUTS = ["summary_table", "baseline", "time_and_failures", "per_task_stats", "paired_tests"]

try:
    from analyze_results import run_analysis  # lives in the parent folder
    from IPython.display import display

    summary = run_analysis(RUN_MODE_NAME, tool="sklearn", outputs=ANALYSIS_OUTPUTS,
                           time_limit=TIME_LIMIT_S)
    display(summary)
except (Exception, SystemExit) as e:
    print(f"[WARN] The analysis could not run ({e}). From a terminal: "
          f"python analyze_results.py {RUN_MODE_NAME} --tool sklearn")
